In [1]:
# ── Config ──────────────────────────────────────────────────────────
RAW_LOG_PATH = "../data/raw/adhi_01.jsonl"   # <-- log to classify
MODEL_DIR = "../model"

# Pick one model to predict with, or set COMPARE_ALL = True to run every
# saved model in MODEL_DIR and compare them side by side.
MODEL_CHOICE = "Random Forest"     # must be a key in MODEL_FILES below
COMPARE_ALL = True

MODEL_FILES = {
    "Naive Bayes": "naive_bayes_model.joblib",
    "K-Nearest Neighbors": "knn_model.joblib",
    "Decision Tree": "decision_tree_model.joblib",
    "Logistic Regression": "logistic_regression_model.joblib",
    "Random Forest": "rf_model.joblib",
    "HistGradientBoosting": "histgb_model.joblib",
    "XGBoost": "xgboost_model.joblib",
}

# ── Imports ─────────────────────────────────────────────────────────
import glob
import json
import os

import joblib
import numpy as np
import pandas as pd

# ── Feature-building functions (same pairing/windowing logic as build_features.py) ──
FLAG_BREAK_BIT = 0x01
FLAG_E0_BIT = 0x02
VK_BACK = 0x08
VK_DELETE = 0x2E
PAUSE_THRESHOLD_MS = 2000
DWELL_CAP_MS = 2000


def load_raw_file(path: str) -> pd.DataFrame:
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    if not rows:
        raise ValueError(f"{path}: no events found")
    df = pd.DataFrame(rows)
    df["source_file"] = os.path.basename(path)
    for col, default in [
        ("seq", np.nan), ("qpc_ticks", np.nan), ("device_handle", ""),
        ("message", np.nan), ("is_autorepeat", np.nan), ("payload_variant", None),
        ("session_id", os.path.basename(path)), ("subject_id", "unknown"), ("label", "unknown"),
    ]:
        if col not in df.columns:
            df[col] = default
    return df


def clean_session(session: pd.DataFrame, log_prefix: str) -> pd.DataFrame:
    before = len(session)
    dedup_cols = [c for c in ["timestamp_ns", "vkey", "scan_code", "flags"] if c in session.columns]
    session = session.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
    n_dupes = before - len(session)

    n_minority_dropped = 0
    if "device_path" in session.columns and session["device_path"].nunique() > 1:
        dominant = session["device_path"].value_counts().idxmax()
        minority_mask = session["device_path"] != dominant
        n_minority_dropped = int(minority_mask.sum())
        session = session[~minority_mask].reset_index(drop=True)

    if n_dupes or n_minority_dropped:
        print(f"  [clean] {log_prefix}: dropped {n_dupes} duplicate events, {n_minority_dropped} minority-device events")
    return session


def pair_dwell_flight(session: pd.DataFrame) -> pd.DataFrame:
    session = session.sort_values("timestamp_ns").reset_index(drop=True)
    open_keys, last_keyup_ts, records = {}, None, []

    for idx, row in session.iterrows():
        is_keyup = bool(int(row["flags"]) & FLAG_BREAK_BIT)
        is_extended = bool(int(row["flags"]) & FLAG_E0_BIT)
        key_id = (row["scan_code"], is_extended)

        if not is_keyup:
            flight_ms = (row["timestamp_ns"] - last_keyup_ts) / 1e6 if last_keyup_ts is not None else np.nan
            open_keys[key_id] = {
                "keydown_ts": row["timestamp_ns"], "vkey": row["vkey"], "scan_code": row["scan_code"],
                "flight_ms": flight_ms, "is_autorepeat": row.get("is_autorepeat", np.nan),
                "device_path": row.get("device_path", None), "seq": row.get("seq", idx),
            }
        else:
            opened = open_keys.pop(key_id, None)
            last_keyup_ts = row["timestamp_ns"]
            if opened is not None:
                dwell_ms = (row["timestamp_ns"] - opened["keydown_ts"]) / 1e6
                records.append({
                    "seq": opened["seq"], "timestamp_ns": opened["keydown_ts"], "vkey": opened["vkey"],
                    "scan_code": opened["scan_code"], "dwell_ms": dwell_ms, "flight_ms": opened["flight_ms"],
                    "is_autorepeat": opened["is_autorepeat"], "device_path": opened["device_path"],
                })
    return pd.DataFrame.from_records(records)


def cv(x: pd.Series) -> float:
    m = x.mean()
    return x.std() / m if np.isfinite(m) and m != 0 else np.nan


def windowize(paired: pd.DataFrame, window: int, step: int, meta: dict) -> pd.DataFrame:
    n = len(paired)
    if n < window:
        return pd.DataFrame()

    rows = []
    for start in range(0, n - window + 1, step):
        w = paired.iloc[start:start + window]
        dwell_all = w["dwell_ms"].dropna()
        dwell_outlier_mask = dwell_all > DWELL_CAP_MS
        dwell = dwell_all[~dwell_outlier_mask]

        flight_all = w["flight_ms"].dropna()
        pause_mask = flight_all > PAUSE_THRESHOLD_MS
        flight = flight_all[~pause_mask]

        span_s = (w["timestamp_ns"].iloc[-1] - w["timestamp_ns"].iloc[0]) / 1e9
        events_per_sec = (len(w) / span_s) if span_s > 0 else np.nan
        backspace_ratio = w["vkey"].isin([VK_BACK, VK_DELETE]).mean()
        autorepeat_ratio = (w["is_autorepeat"].astype(float).mean()
                             if w["is_autorepeat"].notna().any() else np.nan)
        n_devices = w["device_path"].nunique(dropna=True)

        rows.append({
            **meta, "window_start_seq": w["seq"].iloc[0],
            "mean_dwell_ms": dwell.mean(), "std_dwell_ms": dwell.std(), "cv_dwell": cv(dwell),
            "n_dwell_outliers_excluded": int(dwell_outlier_mask.sum()),
            "mean_flight_ms": flight.mean(), "std_flight_ms": flight.std(), "cv_flight": cv(flight),
            "n_pauses_excluded": int(pause_mask.sum()),
            "autorepeat_ratio": autorepeat_ratio, "backspace_ratio": backspace_ratio,
            "events_per_sec": events_per_sec, "n_devices_in_window": n_devices,
        })
    return pd.DataFrame(rows)


def build_windows_for_file(path: str, window: int, step: int) -> pd.DataFrame:
    session = load_raw_file(path)
    session = clean_session(session, os.path.basename(path))
    paired = pair_dwell_flight(session)
    if paired.empty:
        return pd.DataFrame()
    file_meta = {"source_file": session["source_file"].iloc[0], "session_id": session["session_id"].iloc[0]}
    return windowize(paired, window, step, file_meta)


# ── Load shared metadata (feature columns / sentinel / window size — same across all models) ──
with open(os.path.join(MODEL_DIR, "model_meta.json")) as f:
    meta = json.load(f)
feature_cols = meta["feature_columns"]
sentinel = meta["sentinel_fill_value"]
WINDOW = meta["window_size"]
STEP = meta["window_step"]

# ── Build features for the log once, reused for every model ───────
windows = build_windows_for_file(RAW_LOG_PATH, WINDOW, STEP)
if windows.empty:
    raise SystemExit(f"Fewer than {WINDOW} paired keystrokes in this file — not enough data for a verdict.")

X_new = windows[feature_cols].fillna(sentinel)


def run_model(name: str, filename: str):
    model_path = os.path.join(MODEL_DIR, filename)
    if not os.path.exists(model_path):
        print(f"[skip] {name}: {filename} not found in {MODEL_DIR}/")
        return None
    clf = joblib.load(model_path)
    proba = clf.predict_proba(X_new)[:, 1] if hasattr(clf, "predict_proba") else clf.predict(X_new).astype(float)
    pred = (proba >= 0.5).astype(int)
    flagged_rate = pred.mean()
    verdict = "INJECTED" if flagged_rate >= 0.5 else "human"
    return {
        "model": name,
        "n_windows": len(windows),
        "n_flagged": int(pred.sum()),
        "flagged_rate": flagged_rate,
        "mean_proba_injected": proba.mean(),
        "verdict": verdict,
    }


# ── Single model ────────────────────────────────────────────────────
if not COMPARE_ALL:
    result = run_model(MODEL_CHOICE, MODEL_FILES[MODEL_CHOICE])
    print(f"\n=== {os.path.basename(RAW_LOG_PATH)} — {MODEL_CHOICE} ===")
    print(f"  {result['n_windows']} windows evaluated ({WINDOW} keystrokes/window, step {STEP})")
    print(f"  {result['n_flagged']}/{result['n_windows']} windows flagged as injected "
          f"({result['flagged_rate']:.1%})")
    print(f"  mean P(injected) = {result['mean_proba_injected']:.3f}")
    print(f"  VERDICT: {result['verdict']}")

# ── Compare every saved model ──────────────────────────────────────
else:
    print(f"\n=== {os.path.basename(RAW_LOG_PATH)} — comparing all models ===\n")
    results = []
    for name, filename in MODEL_FILES.items():
        r = run_model(name, filename)
        if r:
            results.append(r)

    comparison = pd.DataFrame(results).sort_values("mean_proba_injected", ascending=False)
    print(comparison.to_string(index=False))

    agreement = comparison["verdict"].nunique() == 1
    print(f"\nModels agree: {agreement}")
    if not agreement:
        print("Disagreement — worth a closer look at which features are borderline for this session.")


=== adhi_01.jsonl — comparing all models ===

               model  n_windows  n_flagged  flagged_rate  mean_proba_injected verdict
       Decision Tree        134          5      0.037313         4.542819e-02   human
 Logistic Regression        134          0      0.000000         3.495634e-02   human
 K-Nearest Neighbors        134          0      0.000000         3.198294e-02   human
       Random Forest        134          0      0.000000         2.785100e-02   human
             XGBoost        134          1      0.007463         1.866472e-02   human
HistGradientBoosting        134          1      0.007463         1.208241e-02   human
         Naive Bayes        134          0      0.000000         1.300905e-08   human

Models agree: True
